<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Polygon_Evolution_Animation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Polygon Evolution Animation

### Author Information
- **Author:** Mugambi Ndwiga
- **Social:** [Instagram (@craftsandengineering)](https://www.instagram.com/craftsandengineering)

### Description
This notebook generates a 2D animation of polygons evolving from a triangle (N=3) up to an icosagon (N=20). The visualization is designed with an artistic aesthetic:
- **Fixed Base:** The bottom side of every polygon remains stationary and centered.
- **Constant Scale:** The length of each side remains the same as the number of vertices increases.
- **Paper & Ink Aesthetic:** High-contrast 'bleeding ink' layers for the current shape over a textured paper background.
- **Ghost Outlines:** Previous iterations are preserved as charcoal pencil marks for historical contrast.

### Usage
Simply run the code cell below to generate the interactive JavaScript animation player.

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def create_polygon(n, side_length=1):
    """
    Calculates the vertex coordinates for a regular n-sided polygon.

    The polygon is generated such that the bottom side is perfectly horizontal
    and centered at the bottom of the shape, remaining fixed across different
    values of n. The side length is kept constant to demonstrate growth.

    Args:
        n (int): The number of sides for the polygon.
        side_length (float): The fixed length of each side. Defaults to 1.

    Returns:
        tuple: (x_coords, y_coords) as numpy arrays, including the closing vertex.

    Author: Mugambi Ndwiga (@craftsandengineering)
    """
    # Calculate radius from fixed side length: side = 2 * r * sin(pi/n)
    radius = side_length / (2 * np.sin(np.pi / n))
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False)

    # Rotate to ensure the bottom side is centered at the bottom
    rotation = -np.pi/2 - np.pi/n
    x = radius * np.cos(angles + rotation)
    y = radius * np.sin(angles + rotation)

    # Offset y so the bottom side is always at y=0
    y_offset = -np.min(y)
    return np.append(x, x[0]), np.append(y, y[0]) + y_offset

# Set up the figure and axis with paper aesthetic
fig, ax = plt.subplots(figsize=(10, 10))
ax.set_facecolor('#f4f1ea')
fig.patch.set_facecolor('#f4f1ea')

# Add paper grain texture
x_noise = np.random.rand(150, 150)
ax.imshow(x_noise, extent=[-4, 4, -1, 7], cmap='Greys', alpha=0.07, zorder=0)

ax.set_xlim(-4, 4)
ax.set_ylim(-1, 7)
ax.axis('off')

# Watermark for the animation
ax.text(3.8, -0.8, "Mugambi Ndwiga | @craftsandengineering",
        fontsize=10, color='#222222', alpha=0.5, ha='right', fontfamily='serif')

# Layered lines to simulate bleeding ink
ink_base, = ax.plot([], [], color='#001f3f', lw=6, alpha=0.3, solid_capstyle='round', zorder=10)
ink_core, = ax.plot([], [], color='#000a1a', lw=2, alpha=0.8, solid_capstyle='round', zorder=11)
title_text = ax.text(0, 6.5, "", fontsize=22, ha='center', fontfamily='serif', color='#222222')

# Keep track of ghost line objects to prevent redrawing them unnecessarily during video encoding
ghost_map = {}

def update(frame):
    n_current = int(frame)

    # Ensure ghosts from previous frames are present, but ONLY up to n_current - 1
    # We also explicitly hide any ghosts that belong to future frames to fix the encoding issue
    for n_val in range(3, 21):
        if n_val < n_current:
            if n_val not in ghost_map:
                px, py = create_polygon(n_val)
                line, = ax.plot(px, py, color='#555555', lw=1.2, ls=':', alpha=0.6, zorder=1)
                ghost_map[n_val] = line
            else:
                ghost_map[n_val].set_visible(True)
        elif n_val in ghost_map:
            # Hide lines that exist in the map but shouldn't be visible in this frame
            ghost_map[n_val].set_visible(False)

    # Update the current bleeding ink shape
    x, y = create_polygon(n_current)
    ink_base.set_data(x, y)
    ink_core.set_data(x, y)

    # Dynamic title updates
    names = {3:"Triangle", 4:"Square", 5:"Pentagon", 6:"Hexagon", 8:"Octagon", 10:"Decagon", 12:"Dodecagon", 20:"Icosagon"}
    label = names.get(n_current, f"{n_current}-gon")
    title_text.set_text(label)

    return [ink_base, ink_core, title_text] + list(ghost_map.values())

# Interval set to 2000ms. blit=False is safer for persistent drawings
ani = FuncAnimation(fig, update, frames=range(3, 21), interval=2000, blit=False)

plt.close()
HTML(ani.to_jshtml())

In [15]:
from matplotlib.animation import FFMpegWriter
from google.colab import files

# Re-define the writer to ensure it uses the latest animation state
writer = FFMpegWriter(fps=1, metadata=dict(artist='Mugambi Ndwiga'), bitrate=1800)

# Save the corrected animation
print("Saving updated animation to polygon_evolution.mp4...")
ani.save("polygon_evolution.mp4", writer=writer)
print("Save complete.")

# Trigger the download for the corrected file
files.download("polygon_evolution.mp4")

Saving updated animation to polygon_evolution.mp4...
Save complete.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>